# IND320 project 2026
Philip A. K. Øvland \
Github:     https://github.com/Povland/IND320 \
Streamlit:  https://ind320-2026.streamlit.app/

Video explaining stuff (Please tell me if the link doesn't work): TO DO

## AI Usage

I used AI (Claude) as a helping hand with:
- planning and discussing the strategy before writing code
- looking up the Open-Meteo documentation and API limits
- explaining the methods (DCT, SPC, LOF, STL, spectrogram) and what the parameters do
- first versions of code blocks, which I tested and changed myself, one step at a time
- bug fixing
- giving suggestions to the log, this part, readme and structure

Heavy AI usage with logging helped me remember all the problems I came across, and quicken a somewhat "boring part". It was obviously double checked.

## Log - Tasks
### Post oblig2 work: To make stuff look nicer
- Renamed and reordered the pages with `st.navigation`, wide layout, version in the sidebar (from `pyproject.toml`).
- Home: connection test on load and with a button, data status box, refresh button.
- New About page. Its text is shared with the info boxes on the other pages.
- Cross-border flow: choose connections by country or one by one, several years and months, direction filter, separate views for pie and plot, full-width plot, titles built from the selection.
- Cleanup: comments, imports, README, `.gitignore`, removed duplicates.

### Accounts and repositories
- They are obviously reused.
- Worked on a separate branch (byggeplass) at first, later directly on main.

### API
- Familiarised myself with the Open-Meteo documentation.
- Wrote my own download code with `requests`, since I needed several locations per request, a cache per grid cell and pacing against the limits.

### Strategy change: the app should stand on its own
- At first, the notebook did the work (sampling, download) and the app would only read the results.
- I changed this: all logic now lives in the `ind320` package, and the notebook and the app are two users of the same functions. The notebook explains and tests each step, and the app uses them directly.
- The app runs without the notebook. It computes the grid cells itself from the shapefiles (same seed, same result, checked in the notebook), downloads the weather, and fetches new cross-border data from ENTSO-E.
- All plots that both use are made with plotly, so the same function works in the notebook (`fig.show()`) and in the app (`st.plotly_chart`).

### Shapefiles and areas
- Regions merged into VASS: East = RS + RØ, West = RV, North = RM + RN. NVE_Region needed `encoding="utf-8"` for "RØ".
- `Norge_N5000` includes the sea (`Havflate`), so clipping removed almost nothing until the sea was taken out first.
- The areas are sorted before sampling, so the same seed gives the same points.

### Open-Meteo and the API limit
- One location for a year is about 13 calls, and the free limit is 10,000 per day. My first version (one request per location) took about 25 minutes per area.
- Locations are snapped to the 0.25° ERA5 grid, and each cell is downloaded once. `cell_selection=nearest` and `elevation=nan` make sure the cell asked for is the one returned.
- Each cell is saved as a file as it arrives, so a stopped download continues where it left off. The file is written under a temporary name and renamed when complete, so a stop never leaves a broken file.
- The cache folder is named after the API settings, and a file only counts if it has all the current variables. The MongoDB documents also record their variables and settings, so the app can tell when something has to be downloaded again.
- `snow_depth` is empty for `era5`, so it was removed.
- Open-Meteo sometimes timed out while sending data. The code tries again, and splits the request if it keeps failing.
- 100 locations per area in the notebook, instead of 1000 as the task says. The app can do 100, 200, 500 or 1000, and each number is stored separately. Documents stored before that got `points_per_area = 100` with a small migration.

### Outliers, STL and spectrogram
- SPC on precipitation gives many outliers and none below, since precipitation is skewed and can't be negative. My AI helper said the task text swapped precipitation and temperature. I followed it as written anyway.
- LOF always marks the share set by `contamination`, whatever the data looks like.
- The STL seasonal pattern is asymmetric: the reservoirs fall further below normal in winter than they rise above it in autumn.
- All four functions are tested on made-up series with a known answer.

### Maps
- `ind320.maps`: one area with its grid cells (the dot size is the weight in the average), and all areas of one type coloured by the filling degree.
- The overview map can be played week by week. On a tile map the background jumped for every frame, so it is drawn as a plotly geo map instead, where each frame only recolours the shapes.
- Speed buttons that continue from the current week, a week counter, and colours that follow Streamlit's light and dark theme.
- The static and the animated map use the same layout, so nothing moves when switching. The shapes are prepared once and cached, so the map keeps up when dragging the week slider.

### Streamlit app
- Global choices (type of area, areas, period) are in the sidebar, drawn once in `streamlit_app.py`, and only on the pages that use them. Local choices are on the page.
- Weather Outliers has its own type of area (EL or VASS, since there is no weather polygon for all of Norway), area and year, starting from the sidebar choice. A slider sets the number of locations, and a map shows the grid cells and what the download costs before anything is downloaded.
- One Reservoirs page with three tabs (Map, Plot, Data), replacing the two old reservoir pages.
- "Fetch latest data" at the bottom of every data page: first a free check, then *Download missing* with a progress bar. Home can check all data sources at once, and download weather for several areas and years, with shared cells downloaded once. Downloads bigger than one day of the free API are refused.
- Cross-border data is fetched from ENTSO-E inside the app, straight into MongoDB with pandas, in the same monthly documents as Oblig2. Cassandra and Spark are only in the notebook now.

### Deploying
- The notebook packages (Jupyter, matplotlib, Cassandra, Spark) are in their own dependency group, so Streamlit Cloud only installs what the app needs. Locally: `uv sync --all-groups`.
- Streamlit Cloud installs the `ind320` package as a fixed copy, so after a push with only code changes the app ran old code (AttributeError). Fixed by putting `src/` first on the import path in the app, and rebooting the app after changes in `src/`.
- The ENTSO-E token in the app's secrets had to be in quotes, otherwise the secrets file couldn't be read.

### Still to do
- The current year: ERA5 is about 6 days behind, so it can only be stored up to then, and should be appended day by day.
- Group consecutive SPC outlier hours into events.
- Check whether another weather model has snow depth.

# Setup
Imports used throughout the notebook. The logic lives in the `ind320` package (`src/ind320/`), installed with `uv sync`. The notebook calls it step by step and shows the results, and the Streamlit app uses the same functions.

Plotly figures are shown with the `notebook` renderer, so they are embedded in the HTML export (VS Code's own plotly format is not).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio

from ind320 import areas

# Embeds plotly figures as HTML/JavaScript, so they are kept when the notebook is exported to HTML
pio.renderers.default = "notebook"

DATA_PATH = areas.DATA_PATH

print("Python:", sys.version.split()[0])
print("Running from:", Path.cwd())
print("Data folder:", DATA_PATH)

# Areas and locations
## Shapefiles from NVE Atlas
Downloaded from https://atlas.nve.no on 09.11.2026 (see `data/data_info`):
- `Budområder.zip` (Nettanlegg > Nettområder > Budområder): the price areas NO1–NO5, matching `area_type = EL`
- `NVE_Region.zip` (Administrasjon > Administrative områder > NVE_Region): NVE's five regions, merged into the watercourse areas `area_type = VASS`
- `Norge_N5000.zip` (Administrasjon > Administrative områder > Norge_N5000): land cover for Norway, used to clip the areas to land

| VASS area_nr | Area | NVE regions |
|---|---|---|
| 1 | East | RS + RØ |
| 2 | West | RV |
| 3 | North | RM + RN |

NVE's area list describes VASS 3 as Midt-Norge down to southern Nordland, with VASS 4 for the rest of the north. But VASS 4 has no data (see Oblig2), and NVE's weekly report says area 3 covers Nord-Norge too, so RN goes into VASS 3, as the task says.

Both layers follow watersheds, so the NVE regions are close to the price areas (RØ ≈ NO1, RS ≈ NO2, RV ≈ NO5, RM ≈ NO3, RN ≈ NO4). The main difference is in the north: RM goes up to Saltfjellet, while NO3 stops at Tunnsjødal.

`areas.load_areas()` reads both layers, gives them the same `area_type` and `area_nr` as the reservoir data, and merges the regions. `n_vertices` is the number of points that make up each outline: long coastlines with fjords and islands need many more than inland borders.

In [ ]:
area_table = areas.load_areas()

# Area in km² (the layers are in EPSG:25833, which is in meters)
area_table["km2"] = (area_table.geometry.area / 1e6).round()
area_table.drop(columns="geometry")

## Clipping to land
Both layers go out into the sea along the coast. The weather data is meant to explain reservoir filling, and only precipitation on land can reach a reservoir. Without clipping, the share of points at sea would equal the share of area at sea, which is large for the northern areas, and the region averages would partly describe ocean weather.

`Norge_N5000` is not a land polygon but a land cover map (forest, open land, lakes, glaciers and sea). The sea (`Havflate`) is about a third of it, so it is removed before clipping. Lakes and glaciers are kept, since what falls there reaches the rivers and reservoirs too.

An even closer match would be to sample only inside the catchments that drain to hydropower intakes, but land is a simple and reasonable approximation.

In [ ]:
clipped = areas.clip_to_land(area_table)

# Area and vertices before and after clipping
clipped["km2_clipped"] = (clipped.geometry.area / 1e6).round()
comparison = clipped[["area_type", "area_nr", "km2", "km2_clipped", "n_vertices", "n_vertices_clipped"]].copy()
comparison["share_kept"] = (comparison["km2_clipped"] / comparison["km2"]).round(2)
comparison

## Sampling random locations
The task asks for 1000 locations per area. The notebook uses 100 (the default), and the app lets the user choose 100, 200, 500 or 1000 with a slider. More locations cover an area better, but give more grid cells to download (see the table under *Grid cells*).

Random points are drawn inside each area's bounding box, and only those inside the area are kept. The sampling is done in meters (EPSG:25833), so the points are evenly spread, and the result is converted to latitude and longitude for the weather API. A fixed seed and a fixed order of the areas give the same locations every time.

In [ ]:
locations = areas.sample_locations(clipped)

# Should be 100 per area, 800 in total
print(locations.shape)
locations.groupby(["area_type", "area_nr"]).size()

## Checking the locations on a map
The points should be inside the areas, cover them evenly and stay on land.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

for ax, area_type in zip(axes, ["EL", "VASS"]):
    # Clipped area outlines in black, points coloured by area
    clipped[clipped["area_type"] == area_type].to_crs(4326).boundary.plot(ax=ax, color="black", linewidth=0.5)
    subset = locations[locations["area_type"] == area_type]
    subset.plot(ax=ax, column="area_nr", categorical=True, legend=True, markersize=4)
    ax.set_title(f"{areas.POINTS_PER_AREA} random locations per {area_type} area")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

plt.tight_layout()
plt.show()

## Grid cells
ERA5 is a grid of 0.25° (about 28 × 12 km here). All locations in the same grid cell get the same weather data, so each location is snapped to the nearest grid point. `n_points` says how many of an area's locations fell in each cell, and is used as the cell's weight when averaging over the area.

EL and VASS both cover all of Norway's land, so many cells belong to one EL area and one VASS area. Each cell only needs to be downloaded once.

The app computes the grid cells itself with `areas.grid_cells_for(n)`, for any number of locations, from the same shapefiles and seed. The table below checks that this gives the same cells as the step-by-step version above, and shows how the number of cells grows with the number of locations.

In [ ]:
cells = areas.grid_cells(locations)

per_area = cells.groupby(["area_type", "area_nr"]).agg(cells=("n_points", "size"), points=("n_points", "sum"))
display(per_area)

unique_cells = cells[["latitude", "longitude"]].drop_duplicates()
print(f"{len(cells)} rows (area, cell), {len(unique_cells)} unique grid cells")

In [ ]:
# The app's version (areas.grid_cells_for) should give exactly the same cells as the steps above
assert areas.grid_cells_for(areas.POINTS_PER_AREA).equals(cells), "grid_cells_for doesn't match the step-by-step version"
print("areas.grid_cells_for gives the same cells as the steps above")

# Grid cells per area for each number of locations the app offers. More locations reach more of the area's cells,
# but the number of cells grows much slower than the number of locations
cells_per_option = pd.DataFrame({
    f"{n} locations": areas.grid_cells_for(n).groupby(["area_type", "area_nr"]).size()
    for n in areas.POINT_OPTIONS
})
cells_per_option

# API: Open-Meteo (ERA5)
- API: https://open-meteo.com/en/docs/historical-weather-api
- Data: ERA5 reanalysis, hourly, on a 0.25° grid (~25 km). Licence CC BY 4.0
- Code: `ind320.openmeteo` (the API) and `ind320.weather` (per region, and the MongoDB store)

**Variables:** temperature, precipitation, rain, snowfall and evaporation. Rain reaches the reservoirs right away, snow waits until it melts, and evaporation is water lost. Precipitation is in mm of water, snowfall in cm of snow. `snow_depth` was tried, but comes back empty for the `era5` model, so it was removed.

**The API limit and how it is handled:**
- The free API allows 600 calls per minute, 5,000 per hour and 10,000 per day. A request counts as `n_locations × (n_days / 14) × (n_variables / 10)` calls, so one location for a year with 5 variables is about 13 calls.
- **One request per grid cell, not per location.** All locations in the same ERA5 cell get the same data, so each cell is downloaded once (see *Grid cells* above). `cell_selection=nearest` makes the API use exactly that cell (the default picks a "suitable" land cell nearby), and `elevation=nan` turns off the adjustment of temperature to each point's exact height.
- **Several cells per request** (comma-separated coordinates). This does not save quota, since the weight is multiplied by the number of locations, but gives far fewer requests.
- **Pacing:** each request is filled up to about 500 calls, and the pause after it keeps the download under the minute limit.
- **Resumable:** each cell is saved as a file as soon as it arrives (`data/weather_cells/`, not in git). A new run skips cells that are already there, so a stopped download continues where it left off, and running the notebook again costs no API calls.
- Time is in UTC, like the ENTSO-E data.

In [ ]:
from ind320 import openmeteo, weather

START, END = "2024-01-01", "2024-12-31"

# Cost of one grid cell for the whole year, and of each region
per_cell = openmeteo.call_weight(1, START, END)
cost = cells.groupby(["area_type", "area_nr"]).size().rename("cells").to_frame()
cost["api_calls"] = (cost["cells"] * per_cell).round()

print(f"One grid cell for {START} to {END}: about {per_cell:.1f} API calls")
print(f"All regions, each cell once: {len(unique_cells)} cells, about {len(unique_cells) * per_cell:,.0f} API calls")
cost

## The download function
`weather.download_region_weather(region, start, end)` takes a region identifier (`"NO1"`–`"NO5"` or `"VASS1"`–`"VASS3"`) and a time interval, looks up the region's grid cells (latitude and longitude pairs, computed from the shapefiles with `areas.grid_cells_for`, see *Grid cells*)

Each cell is weighted by `n_points`, the number of the region's random locations that fell in it. This gives the same result as averaging over all the locations, without downloading the same cell more than once. Only cells with a value count in the weights, so a missing value doesn't pull the average down.

## NO3 for 2024

In [ ]:
REGION = "NO3"
POINTS = areas.POINTS_PER_AREA   # 100 locations, the default

# Downloads NO3 for 2024, or reads the cells from data/weather_cells/
no3 = weather.download_region_weather(REGION, START, END, points_per_area=POINTS)

# 2024 is a leap year: 366 days x 24 hours = 8784 hours
print(no3.shape)
print("Period:", no3.index.min(), "to", no3.index.max())
assert len(no3) == 366 * 24, f"Expected 8784 hours, got {len(no3)}"

# Missing values per variable (should be 0 for all)
no3.isna().sum()

## Storing in MongoDB
The region average is stored in MongoDB, one document per region, year and number of locations (`points_per_area`), which is where the Streamlit app reads it. Each document also records its variables and the API settings, so the app can tell when it must be downloaded again. Storing the same region, year and number again replaces the document instead of adding a new one.

The connection uses the same credentials file as Oblig2 (`No_sync/atlas-credentials.env`, not in git).

In [ ]:
from pymongo import MongoClient

from ind320.config import load_env

# Connects to MongoDB Atlas with the credentials from No_sync/atlas-credentials.env
creds = load_env("atlas-credentials.env")
mongo_client = MongoClient(creds["MONGODB_URI"], serverSelectionTimeoutMS=10000)
mongo_client.admin.command("ping")
mongo_db = mongo_client["ind320"]
print("Connected to MongoDB")

# Documents stored before the number of locations was part of the key get points_per_area = 100.
# Safe to run every time: it only changes documents that lack the field
print(weather.migrate_weather_documents(mongo_db), "old documents updated")

# Stores NO3 2024, reads it back and checks that nothing changed on the way
weather.save_region_weather(mongo_db, REGION, no3, points_per_area=POINTS)
from_mongo = weather.load_region_weather(mongo_db, REGION, 2024, points_per_area=POINTS)
assert from_mongo.equals(no3), "The data read back from MongoDB is not the same as what was stored"
print(f"{REGION} 2024 ({POINTS} locations) stored and read back: {from_mongo.shape}")

# Everything stored so far
weather.stored_weather(mongo_db)

## Sanity check
Daily mean temperature and daily precipitation over NO3. Temperature should go from below zero in winter to around 10–15 °C in summer. This is also the first plotly figure, to check that it shows in the HTML export.

In [ ]:
from plotly.subplots import make_subplots

daily_temperature = no3["temperature_2m"].resample("1D").mean()
daily_precipitation = no3["precipitation"].resample("1D").sum()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Daily mean temperature", "Daily precipitation"])
fig.add_scatter(x=daily_temperature.index, y=daily_temperature, name="Temperature", row=1, col=1)
fig.add_bar(x=daily_precipitation.index, y=daily_precipitation, name="Precipitation", row=2, col=1)
fig.update_yaxes(title_text="°C", row=1, col=1)
fig.update_yaxes(title_text="mm", row=2, col=1)
fig.update_layout(title=f"{REGION} {START[:4]}: average over {len(weather.region_cells(REGION))} grid cells",
                  height=600, showlegend=False)
fig.show()

# Outliers and anomalies
The functions are in `ind320.anomalies`, so the Streamlit app can use them too. Both return `(figure, summary, outliers)`: a plotly figure, a summary of the outliers, and a table with one row per outlier.

## Precipitation: DCT high-pass filter and SPC
**High-pass filter.** The Discrete Cosine Transform (DCT) writes the series as a sum of cosine waves. Low coefficients are slow waves (season, trend), high coefficients are fast variations (weather). Coefficient *k* has a period of 2N/k samples, so with N = 8784 hours and a cut-off of 30 days (720 hours), every coefficient with k < 2 · 8784 / 720 ≈ 24.4 is removed. What is left after transforming back is the seasonally adjusted variation (SATV).

**SPC boundaries.** The boundaries are the median of the SATV ± `n_std` robust standard deviations, estimated from the whole year. The robust standard deviation is 1.4826 × MAD (median absolute deviation), which equals the standard deviation for normally distributed data. The median and MAD are used instead of the mean and standard deviation, since outliers make the standard deviation larger and so hide themselves (masking).

**Curves, not lines.** The boundaries are computed on the SATV, but the original series is plotted, so the slow part removed by the filter is added back to the boundaries. That is why they follow the season. The SATV itself is not plotted.

**Parameters:** `cutoff_days` (default 30) and `n_std` (default 3, the classic SPC limit).

**Note on the task text:** it says "high-pass filtering of the precipitation" to create "seasonally adjusted *temperature* variations". The task is followed as written (precipitation), and the results below show why the method fits precipitation poorly.

In [ ]:
from ind320 import anomalies

fig, spc_summary, spc_outliers = anomalies.spc_outliers(
    no3["precipitation"], title=f"{REGION} 2024: hourly precipitation with SPC outliers", unit="mm"
)
fig.show()
display(spc_summary)
spc_outliers.head(10)

### What the SPC result shows
- **Far more outliers than SPC expects.** With normally distributed variation, ±3 standard deviations would mark about 0.27 % of the hours. Here it is several percent, because precipitation is not normally distributed: most hours are dry (0 mm), and a few have heavy rain, which gives a long tail upwards.
- **No outliers below.** The lower boundary is negative, and precipitation can never be. So for precipitation the method only finds heavy rain, never "unusually little rain".
- **The median of the SATV is negative.** Most hours are dry, and 0 minus a positive seasonal level is negative. This is another sign of the skewed distribution.
- **One rain event gives several outliers in a row.** A heavy rain over several hours counts as several outliers. A possible improvement would be to group consecutive outlier hours into events, and count events instead of hours.

SPC with symmetric boundaries assumes variation that is roughly normally distributed around a level. That fits temperature well and precipitation poorly, which supports the idea that the variables in the task text were swapped.

## Temperature: Local Outlier Factor
LOF compares how densely packed each point's neighbourhood is with how densely packed its neighbours' neighbourhoods are. A point in a sparser area than its neighbours gets a high score (`lof_score`) and is marked as an anomaly.

**Features.** LOF works on points in a space, not on a time series as such. Here the features are time and temperature, both standardized (so that one day and one degree are comparable), which means a temperature is compared with temperatures close in time. With temperature alone, LOF would only find the rarest temperatures of the whole year.

**Parameters:** `contamination`, the share of points marked as anomalies (default 1 %, as the task says), and `n_neighbors` (default 20).

In [ ]:
fig, lof_summary, lof_outliers = anomalies.lof_anomalies(
    no3["temperature_2m"], title=f"{REGION} 2024: hourly temperature with LOF anomalies", unit="°C"
)
fig.show()
display(lof_summary)
lof_outliers.head(10)

### What the LOF result shows
- **Always 1 %.** LOF marks the share given by `contamination`, whatever the data looks like. It ranks the points, but does not decide whether anything is unusual. SPC, in contrast, can find no outliers at all if nothing crosses the boundaries.
- **Unusual for its time, not extreme for the year.** The highest scores are temperatures that stand out from the hours around them, not necessarily the coldest or warmest of the year.
- **Anomalies come in groups.** Like the rain events above, a warm or cold spell gives several anomalies in a row.

## Testing the functions
1. **Known answer:** a made-up hourly series for one year (a seasonal wave, a daily wave and noise) with three spikes added at known hours. Both methods should find all three.
2. **Parameters:** the number of outliers should change in the expected direction when `n_std` and `contamination` change.

In [ ]:
# Made-up series: seasonal wave + daily wave + noise, with three spikes at known hours
rng = np.random.default_rng(320)
hours = pd.date_range("2024-01-01", periods=366 * 24, freq="h", tz="UTC")
t = np.arange(len(hours))
synthetic = pd.Series(
    -10 * np.cos(2 * np.pi * t / len(t)) + 3 * np.sin(2 * np.pi * t / 24) + rng.normal(0, 1, len(t)),
    index=hours,
)
spike_hours = hours[[1000, 4000, 7000]]
synthetic[spike_hours] += [15, -15, 15]

# SPC should find all three spikes, above and below
_, summary, found = anomalies.spc_outliers(synthetic)
assert spike_hours.isin(found.index).all(), "SPC missed a spike"
print(f"SPC found all three spikes ({summary['n_outliers']} outliers in total)")

# LOF should find them too
_, summary, found = anomalies.lof_anomalies(synthetic)
assert spike_hours.isin(found.index).all(), "LOF missed a spike"
print(f"LOF found all three spikes ({summary['n_outliers']} anomalies in total)")

# LOF marks the share given by contamination
assert abs(summary["share_outliers"] - 0.01) < 0.002
print("LOF marked 1 % of the values, as set by contamination")

In [ ]:
# Parameters: more standard deviations -> fewer SPC outliers, higher contamination -> more LOF anomalies
spc_counts = [anomalies.spc_outliers(no3["precipitation"], n_std=n)[1]["n_outliers"] for n in [2, 3, 4]]
print("SPC precipitation, n_std = 2, 3, 4:", spc_counts)
assert spc_counts[0] > spc_counts[1] > spc_counts[2]

for cutoff_days in [7, 30, 90]:
    summary = anomalies.spc_outliers(no3["precipitation"], cutoff_days=cutoff_days)[1]
    print(f"SPC precipitation, cutoff_days = {cutoff_days}: {summary['n_outliers']} outliers "
          f"({summary['dct_coefficients_removed']} coefficients removed)")

lof_counts = [anomalies.lof_anomalies(no3["temperature_2m"], contamination=c)[1]["n_outliers"] for c in [0.005, 0.01, 0.05]]
print("LOF temperature, contamination = 0.5 %, 1 %, 5 %:", lof_counts)
assert lof_counts[0] < lof_counts[1] < lof_counts[2]

print("All parameter tests passed")

# Seasonal-Trend decomposition using LOESS (STL)
The functions are in `ind320.seasonal`. The data is the weekly filling degree from NVE's Magasinstatistikk API, for the same area as the weather data (NO3).

STL splits the series into three parts: **observed = trend + seasonal + residual**. Each part is estimated with LOESS (local smoothing), and the parameters set how wide the smoothing windows are:

| Parameter | Default | Unit | What it controls |
|---|---|---|---|
| `period` | 52 | weeks | The length of one season (fixed, since the data is weekly) |
| `seasonal` | 13 | years | How many years the seasonal value for a given week is smoothed over. Small lets the seasonal pattern change from year to year, large keeps it almost fixed. Must be odd |
| `trend` | 105 | weeks | How wide the trend is smoothed (about 2 years). Must be longer than the period, otherwise the trend takes over the seasonal cycle. Must be odd |
| `robust` | True | | Down-weights unusual weeks, so a single dry year does not shift what counts as normal |

`seasonal` counts years and `trend` counts weeks, because the seasonal smoother runs along the same week in each year (one value per year), while the trend smoother runs along the series week by week.

A year is 52.18 weeks on average, but STL needs a whole number, so 52 is used.

In [ ]:
from ind320 import nve, seasonal

reservoirs = nve.fetch_reservoir_data()
filling = seasonal.filling_series(reservoirs, "EL", 3)
print(f"{len(filling)} weeks from {filling.index.min():%Y-%m-%d} to {filling.index.max():%Y-%m-%d}, "
      f"{filling.isna().sum()} missing")

fig, components = seasonal.stl_decomposition(filling, title="NO3: STL decomposition of the filling degree")
fig.show()

# How much of the variation each part has
components.std().round(3)

# Spectrogram
The spectrogram shows which frequencies are strong, and when. The series is cut into overlapping windows, the power at each frequency is found for each window, and the windows are put side by side: time along the x axis, frequency (cycles per year) on the y axis, and power as colour (in dB). The yearly cycle is at 1 cycle per year.

| Parameter | Default | What it controls |
|---|---|---|
| `window_length` | 156 weeks (3 years) | The frequency resolution is samples per year / window length, here 52.18 / 156 ≈ 0.33 cycles per year. A longer window gives sharper frequencies but blurs *when* things change |
| `window_overlap` | 117 weeks (75 %) | How much the windows share. Each new window starts 39 weeks after the previous one, which gives a smooth picture over time |

A window of one year would be a poor choice: with a resolution of about 1 cycle per year, the yearly cycle could not be told apart from 0 or 2.

Each window's mean is removed before the power is found, so the level of the filling degree does not show up as a strong frequency at 0. Frequencies above 6 cycles per year are left out of the plot, since they are mostly noise.

In [ ]:
fig, power = seasonal.spectrogram(filling, title="NO3: spectrogram of the filling degree")
fig.show()

print(f"{power.shape[0]} frequencies x {power.shape[1]} windows")
print(f"Strongest frequency on average: {power.mean(axis=1).idxmax():.2f} cycles per year")

## Testing the functions
1. **STL, known answer:** a made-up weekly series with a known trend and seasonal wave. STL should recover both closely.
2. **STL, invalid parameters** should give a clear error, not a crash later on.
3. **Spectrogram, known answer:** a made-up series with a wave at 2 cycles per year. The strongest frequency should be 2.
4. **Spectrogram, parameters:** a longer window should give more (finer) frequencies and fewer windows.

In [ ]:
# Made-up weekly series for 30 years: a slow trend + a yearly wave + noise
rng = np.random.default_rng(320)
weeks = pd.date_range("1995-01-08", periods=30 * 52, freq="W-SUN")
t = np.arange(len(weeks))
known_trend = 0.6 + 0.1 * np.sin(2 * np.pi * t / (8 * 52))      # an 8-year wave
known_seasonal = 0.25 * np.sin(2 * np.pi * t / 52)
synthetic = pd.Series(known_trend + known_seasonal + rng.normal(0, 0.02, len(t)), index=weeks)

# 1. STL should recover the trend and the seasonal part
_, found = seasonal.stl_decomposition(synthetic)
trend_corr = np.corrcoef(found["trend"], known_trend)[0, 1]
seasonal_corr = np.corrcoef(found["seasonal"], known_seasonal)[0, 1]
print(f"Correlation with the known parts: trend {trend_corr:.3f}, seasonal {seasonal_corr:.3f}")
assert trend_corr > 0.95 and seasonal_corr > 0.95

# 2. Invalid parameters give a clear error
for bad in [dict(seasonal=12), dict(trend=51), dict(trend=104)]:
    try:
        seasonal.stl_decomposition(synthetic, **bad)
        raise AssertionError(f"No error for {bad}")
    except ValueError as error:
        print(f"{bad}: {error}")

print("STL tests passed")

In [ ]:
# 3. A wave at 2 cycles per year should give the strongest frequency at 2
two_per_year = pd.Series(np.sin(2 * np.pi * 2 * t / seasonal.WEEKS_PER_YEAR) + rng.normal(0, 0.1, len(t)), index=weeks)
_, found = seasonal.spectrogram(two_per_year)
strongest = found.mean(axis=1).idxmax()
print(f"Strongest frequency: {strongest:.2f} cycles per year")
assert abs(strongest - 2) < 0.2

# 4. Longer window -> more frequencies (finer resolution) and fewer windows
shapes = {length: seasonal.spectrogram(filling, window_length=length, window_overlap=length * 3 // 4)[1].shape
          for length in [104, 156, 260]}
for length, (n_frequencies, n_windows) in shapes.items():
    print(f"window_length = {length} weeks: {n_frequencies} frequencies, {n_windows} windows")
assert shapes[104][0] < shapes[156][0] < shapes[260][0]
assert shapes[104][1] > shapes[156][1] > shapes[260][1]

print("Spectrogram tests passed")

# Maps
The functions are in `ind320.maps`, and are used on the Weather Outliers and Reservoirs pages in the app. The outlines are simplified before drawing, since large GeoJSON makes the maps slow.

`area_map` shows one area with its grid cells on an open map background, sized by how many random locations fell in each (the weights in the average). `overview_map` shows all areas of one type coloured by a value, with chosen areas outlined. It is drawn as a plotly geo map without background tiles, which is what lets the app play it week by week (`overview_animation`): each frame only recolours the shapes.

In [ ]:
from ind320 import maps

land = areas.land_areas()
no3_area = land[(land["area_type"] == "EL") & (land["area_nr"] == 3)]

# NO3 with its grid cells, for 100 and 1000 locations
for n in [100, 1000]:
    no3_cells = weather.region_cells("NO3", points_per_area=n)
    maps.area_map(no3_area, no3_cells, title=f"NO3: {n} locations in {len(no3_cells)} grid cells").show()

In [ ]:
# The price areas coloured by the latest filling degree, with NO3 outlined
latest_week = reservoirs.index.max()
latest = reservoirs.loc[[latest_week]]
latest_el = latest[latest["area_type"] == "EL"].set_index("area_nr")["filling_degree"]

price_areas = land[land["area_type"] == "EL"]
maps.overview_map(
    price_areas, latest_el, selected=[3], label="Filling degree", zmin=0, zmax=1,
    title=f"Filling degree per price area, week ending {latest_week:%d.%m.%Y}",
).show()